# 通过 Drive API 创建 SAFE 文件个人副本（V4）

快捷方式菜单中的“复制”只会复制快捷方式。本 Notebook 直接调用 Google Drive `files.copy`，尝试把原始 ZIP 复制到 `我的云端硬盘/SAFE-rollouts`。默认处理 Pi0-FAST；完成上传并释放 Drive 空间后，可把配置改成第二组 OpenVLA/WidowX。

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
drive_service = build('drive', 'v3', cache_discovery=False)
print('Google Drive API 授权完成。')

In [ ]:
# 默认复制第一份。处理第二份时，把下面两行替换为注释中的值。
SOURCE_ID = '13z_cdwnaJota2iHkZbhYgVALujZwtM3b'
DEST_NAME = 'pi0fast_droid_0510_all.zip'
# SOURCE_ID = '1EwaccasZjnlM9L6SEYyWqTd7d6-BR9zp'
# DEST_NAME = 'openvla_widowx.zip'

FOLDER_NAME = 'SAFE-rollouts'
folder_query = (
    f"name = '{FOLDER_NAME}' and "
    "mimeType = 'application/vnd.google-apps.folder' and "
    "'root' in parents and trashed = false"
)
folders = drive_service.files().list(
    q=folder_query, spaces='drive', fields='files(id,name)'
).execute().get('files', [])
if folders:
    folder_id = folders[0]['id']
else:
    folder = drive_service.files().create(
        body={
            'name': FOLDER_NAME,
            'mimeType': 'application/vnd.google-apps.folder',
        },
        fields='id,name',
    ).execute()
    folder_id = folder['id']
print(f'目标文件夹：我的云端硬盘/{FOLDER_NAME} ({folder_id})')

metadata = drive_service.files().get(
    fileId=SOURCE_ID,
    fields='id,name,size,copyRequiresWriterPermission,capabilities(canCopy,canDownload)',
    supportsAllDrives=True,
).execute()
print('原文件信息：', metadata)
if not metadata.get('capabilities', {}).get('canCopy', False):
    raise PermissionError('文件所有者未授权复制；只能等待配额重置或请求作者提供镜像。')

quota = drive_service.about().get(fields='storageQuota').execute()['storageQuota']
source_size = int(metadata.get('size', 0))
if quota.get('limit'):
    free_bytes = int(quota['limit']) - int(quota.get('usage', 0))
    print(f'个人 Drive 可用空间：{free_bytes:,} bytes；文件：{source_size:,} bytes')
    if source_size and free_bytes < source_size:
        raise OSError('个人 Google Drive 空间不足，无法创建该文件副本。')

existing_query = (
    f"name = '{DEST_NAME}' and '{folder_id}' in parents and "
    "trashed = false"
)
existing = drive_service.files().list(
    q=existing_query, spaces='drive', fields='files(id,name,size,ownedByMe)'
).execute().get('files', [])
owned = [item for item in existing if item.get('ownedByMe')]
if owned:
    result = owned[0]
    print('已存在个人副本，跳过重复创建：', result)
else:
    try:
        result = drive_service.files().copy(
            fileId=SOURCE_ID,
            body={'name': DEST_NAME, 'parents': [folder_id]},
            fields='id,name,size,md5Checksum,ownedByMe,parents',
            supportsAllDrives=True,
        ).execute()
    except HttpError as exc:
        raise RuntimeError(
            'Drive API 无法创建副本。若错误仍是 downloadQuotaExceeded，'
            '则只能等待配额重置或请求作者提供镜像。'
        ) from exc
    print('个人副本创建完成：', result)

print('下一步：打开 V3 Notebook，挂载 Drive 并上传服务器。')